In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

import os, pickle, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib, matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    print('scienceplots not installed; using default style')

from plot_violin import compute_jaccard_distance
import baseline as baseline_mod

matplotlib.rcParams.update({
    'font.size':       12,
    'axes.labelsize':  13,
    'axes.titlesize':  13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.titlesize':14,
})

DS          = 'arrhythmia'
MODELS      = ['mlp', 'ftt']
METHODS     = ['random', 'cte']
METHOD_LABELS  = {'random': 'Random', 'cte': 'CTE'}
METHOD_COLORS  = {'random': '#4393C3', 'cte': '#9970AB'}
MODEL_LABELS   = {'mlp': 'MLP', 'ftt': 'FTT'}

D_NOMINAL   = 278          # full feature space (l1_reg acts on this)
JAC_K       = 3
N_BG_SEEDS  = 5
JAC_VALUES  = [0.0, 0.5, 0.8, 1.0]
JAC_COLORS  = {0.0: '#2b8a3e', 0.5: '#ffd43b', 0.8: '#fd7e14', 1.0: '#e03131'}
JAC_LABELS  = {0.0: 'Jac=0 (no change)', 0.5: 'Jac=0.5 (1 feature)',
               0.8: 'Jac=0.8 (2 features)', 1.0: 'Jac=1.0 (all different)'}
PAIRS_10    = list(itertools.combinations(range(N_BG_SEEDS), 2))

RES_DIR  = 'results_tree'
FIG_DIR  = 'figures_b/arrhythmia_analysis'
os.makedirs(FIG_DIR, exist_ok=True)
print('Config loaded.')


In [ ]:
# Load all arrhythmia SHAP arrays
# sv_data[model][method] = list of 5 arrays (N_test=84, D_NOMINAL=278)

sv_data = {mo: {} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs = []
        for s in range(N_BG_SEEDS):
            path = os.path.join(RES_DIR, f'{DS}_{model}_0_0_{method}_s{s}_sv.pkl')
            with open(path, 'rb') as f:
                obj = pickle.load(f)
            arr = obj.values if hasattr(obj, 'values') else obj['values']
            svs.append(arr)
        sv_data[model][method] = svs
        print(f'{model}/{method}: 5 seeds, shape {svs[0].shape}')

print('\nAll data loaded.')


## 1. Baseline (reference only — l1_reg caveat)

In [ ]:
# ── 1. Baseline (reference only) ──────────────────────────────────────────────
# l1_reg="num_features(10)" forces 268/278 features to exactly 0,
# so the d=278 Mallows null does NOT match the actual ranking structure
# (effective ranking is over ~10 features, not 278).
# Baseline is computed for completeness; exceedance rate is labelled REFERENCE ONLY.

print('Computing Jaccard baseline at d=278, r=3, q in {0.3, 0.4, 0.5} ...')
jac_band = {}
for q in [0.3, 0.4, 0.5]:
    res = baseline_mod.compute_jaccard_baseline(d=D_NOMINAL, r=JAC_K, q=q, n_trials=20000, seed=0)
    jac_band[q] = res['mean']
    print(f'  q={q}: E[Jac@3] = {res["mean"]:.4f}')

tau_jac     = min(jac_band.values())
tau_jac_max = max(jac_band.values())
print(f'\nJaccard band: [{tau_jac:.4f}, {tau_jac_max:.4f}]  (tau = {tau_jac:.4f})')
print('  NOTE: at d=278, random top-3 pairs almost never overlap -> tau near 1.0')
print('  Actual data with ~10 non-zero features will mostly fall BELOW this band.')

# L2 Dirichlet baseline with empirical T
print('\nComputing L2 baseline ...')
all_shap = np.concatenate([
    np.abs(sv_data[mo][mt][s])
    for mo in MODELS for mt in METHODS for s in range(N_BG_SEEDS)
])
T_emp = float(np.percentile(all_shap.max(axis=1), 95))
print(f'  Empirical T (95th-pct of max|SHAP|) = {T_emp:.4f}')

from baseline import compute_l2_baseline as _cl2
R_EFF = max(1, D_NOMINAL // 3)
l2_ref = _cl2(D_NOMINAL, R_EFF, T=T_emp)
print(f'  L2 baseline (d={D_NOMINAL}, r={R_EFF}, T={T_emp:.4f}): {l2_ref:.4f}')
print('\n[All above values are REFERENCE ONLY — l1_reg caveat applies]')

# Actual pairwise Jac distributions for exceedance comparison
all_jac_flat = np.concatenate([
    compute_jaccard_distance(sv_data[mo][mt][i][:], sv_data[mo][mt][j][:], k=JAC_K)
    for mo in MODELS for mt in METHODS for (i, j) in PAIRS_10
])
pct_above = float(np.mean(all_jac_flat > tau_jac))
print(f'  Pct pairs > tau_jac={tau_jac:.4f}: {pct_above:.1%}  [REFERENCE ONLY]')


## 2. Non-zero feature distribution & support overlap (l1_reg sanity check)

In [ ]:
# ── 2a. Sanity check: non-zero feature count per (instance, seed) ─────────────
EPS = 1e-10

support_sizes = {mo: {mt: [] for mt in METHODS} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs = sv_data[model][method]
        for s in range(N_BG_SEEDS):
            sz = (np.abs(svs[s]) > EPS).sum(axis=1)  # (N_test,)
            support_sizes[model][method].append(sz)
        all_sz = np.concatenate(support_sizes[model][method])
        print(f'{model}/{method:<8}: non-zero/instance  '
              f'mean={all_sz.mean():.2f}  median={int(np.median(all_sz))}  '
              f'min={all_sz.min()}  max={all_sz.max()}  '
              f'pct_exactly10={np.mean(all_sz==10):.1%}')

print('\n(l1_reg="num_features(10)" expected: nearly all instances have exactly 10 non-zero)')


In [ ]:
# ── 2b. Support set pairwise Jaccard ─────────────────────────────────────────
EPS = 1e-10

print('Support-set pairwise Jaccard (10 pairs x 84 instances):\n')
support_jac = {mo: {mt: None for mt in METHODS} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs = sv_data[model][method]
        min_n = min(len(v) for v in svs)
        vals = []
        for (i, j) in PAIRS_10:
            sup_i = np.abs(svs[i][:min_n]) > EPS
            sup_j = np.abs(svs[j][:min_n]) > EPS
            inter = (sup_i & sup_j).sum(axis=1).astype(float)
            union = (sup_i | sup_j).sum(axis=1).astype(float)
            jac   = np.where(union > 0, inter / union, 1.0)
            vals.append(jac)
        all_vals = np.concatenate(vals)
        support_jac[model][method] = all_vals
        print(f'{model}/{method:<8}: support Jac  '
              f'mean={all_vals.mean():.3f}  '
              f'median={np.median(all_vals):.3f}  '
              f'pct_identical={np.mean(all_vals==1.0):.1%}  '
              f'pct_zero={np.mean(all_vals==0.0):.1%}')


## 3. Null-free core metrics

In [ ]:
# ── 3. Top-1 churn: any-seed + pairwise ──────────────────────────────────────

top1_any  = {mo: {} for mo in MODELS}
top1_pair = {mo: {} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs   = sv_data[model][method]
        min_n = min(len(v) for v in svs)

        top1s     = [np.argmax(np.abs(svs[s][:min_n]), axis=1) for s in range(N_BG_SEEDS)]
        pair_arr  = np.concatenate([(top1s[i] != top1s[j]).astype(float) for i, j in PAIRS_10])
        top1_pair[model][method] = float(pair_arr.mean())

        top1_mat     = np.stack(top1s, axis=0)  # (5, N)
        any_disagree = ~np.all(top1_mat == top1_mat[[0], :], axis=0)
        top1_any[model][method] = float(any_disagree.mean())

print('Top-1 churn:')
print(f'{"":12}  {"any-seed":>10}  {"pairwise":>10}')
for mo in MODELS:
    for mt in METHODS:
        print(f'{mo}/{mt:<8}:  {top1_any[mo][mt]:>9.1%}  {top1_pair[mo][mt]:>9.1%}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
fig.suptitle('Arrhythmia — Top-1 Feature Churn', fontsize=13, fontweight='bold')

for ax, churn_dict, label in zip(
        axes,
        [top1_any, top1_pair],
        ['Any-seed (≥1 disagrees)', 'Pairwise (10N comparisons)']):
    combos = [(mo, mt) for mo in MODELS for mt in METHODS]
    labels = [f'{MODEL_LABELS[mo]}\n{METHOD_LABELS[mt]}' for mo, mt in combos]
    vals   = [churn_dict[mo][mt] for mo, mt in combos]
    colors = [METHOD_COLORS[mt] for _, mt in combos]

    x = np.arange(len(combos))
    ax.bar(x, vals, color=colors, edgecolor='white', linewidth=0.5)
    for xi, v in enumerate(vals):
        ax.text(xi, v + 0.005, f'{v:.1%}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, max(vals) * 1.35 + 0.02)
    ax.set_ylabel('Fraction of instances' if churn_dict is top1_any else 'Fraction of pair-instance comparisons')
    ax.set_title(label, fontsize=11)

handles = [mpatches.Patch(color=METHOD_COLORS[m], label=METHOD_LABELS[m]) for m in METHODS]
fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.9)
fig.tight_layout(rect=[0, 0.08, 1, 1])
fname = os.path.join(FIG_DIR, 'arr_top1_churn.pdf')
fig.savefig(fname, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved: {fname}')


In [ ]:
# ── 4. Top-3 Jaccard proportions ─────────────────────────────────────────────

def round_jac(val):
    return min(JAC_VALUES, key=lambda x: abs(x - val))

jac_props = {mo: {} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs   = sv_data[model][method]
        min_n = min(len(v) for v in svs)
        all_jac = np.concatenate([
            compute_jaccard_distance(svs[i][:min_n], svs[j][:min_n], k=JAC_K)
            for i, j in PAIRS_10
        ])
        counts = {v: 0 for v in JAC_VALUES}
        for val in all_jac:
            counts[round_jac(float(val))] += 1
        total = len(all_jac)
        jac_props[model][method] = {v: counts[v]/total for v in JAC_VALUES}

print('Top-3 Jaccard proportions:')
print(f'{"":14}  {"Jac=0":>7}  {"Jac=0.5":>7}  {"Jac=0.8":>7}  {"Jac=1.0":>7}')
for mo in MODELS:
    for mt in METHODS:
        p = jac_props[mo][mt]
        print(f'{mo}/{mt:<8}:  {p[0.0]:>7.1%}  {p[0.5]:>7.1%}  {p[0.8]:>7.1%}  {p[1.0]:>7.1%}')

# ── Plot stacked bar ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
fig.suptitle('Arrhythmia — Discrete Top-3 Jaccard Distribution', fontsize=13, fontweight='bold')

for ax, model in zip(axes, MODELS):
    x       = np.arange(len(METHODS))
    bottoms = np.zeros(len(METHODS))
    for jv in JAC_VALUES:
        heights = np.array([jac_props[model][m][jv] for m in METHODS])
        ax.bar(x, heights, bottom=bottoms, color=JAC_COLORS[jv], edgecolor='white', linewidth=0.5)
        for xi, (h, b) in enumerate(zip(heights, bottoms)):
            if h > 0.04:
                txt_color = 'white' if jv == 1.0 else 'black'
                ax.text(xi, b + h/2, f'{h:.2%}', ha='center', va='center',
                        fontsize=8, color=txt_color, fontweight='bold')
        bottoms += heights
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[m] for m in METHODS])
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Proportion of (pair, instance)')
    ax.set_title(MODEL_LABELS[model])

handles = [mpatches.Patch(color=JAC_COLORS[jv], label=JAC_LABELS[jv]) for jv in JAC_VALUES]
fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.06), framealpha=0.9)
fig.tight_layout(rect=[0, 0.12, 1, 1])
fname = os.path.join(FIG_DIR, 'arr_jaccard_stacked.pdf')
fig.savefig(fname, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved: {fname}')


## 4. Consensus top-3 rank-position heatmap

In [ ]:
# ── 5. Consensus top-3 rank-position heatmap (x clipped to rank 1-15) ────────
RANK_CLIP = 15

rank_dist = {mo: {} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs    = sv_data[model][method]
        min_n  = min(len(v) for v in svs)
        abs_sv = np.stack([np.abs(v[:min_n]) for v in svs], axis=0)  # (5, N, F)

        mean_abs       = abs_sv.mean(axis=0)                              # (N, F)
        consensus_top3 = np.argsort(-mean_abs, axis=1)[:, :3]            # (N, 3)
        ranks          = np.argsort(np.argsort(-abs_sv, axis=2), axis=2) # (5, N, F) 0-idx

        rank_dist[model][method] = {}
        for c in range(3):
            feat_idx = consensus_top3[:, c]  # (N,)
            rpos = np.array([
                ranks[s, np.arange(min_n), feat_idx] + 1  # 1-indexed
                for s in range(N_BG_SEEDS)
            ]).flatten()
            rank_dist[model][method][c] = rpos

# 2×2 grid: model (rows) × method (cols)
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
fig.suptitle(
    f'Arrhythmia — Rank Positions of Consensus Top-3 Features\n'
    f'(x-axis clipped to top-{RANK_CLIP}; cell = fraction of seed observations)',
    fontsize=12, fontweight='bold'
)

for row, model in enumerate(MODELS):
    for col, method in enumerate(METHODS):
        ax = axes[row][col]
        rd = rank_dist[model][method]

        heatmap = np.zeros((3, D_NOMINAL))
        for c in range(3):
            rpos  = rd[c]
            total = len(rpos)
            for r in range(1, D_NOMINAL + 1):
                heatmap[c, r - 1] = np.sum(rpos == r) / total

        clipped = heatmap[:, :RANK_CLIP]
        im = ax.imshow(clipped, aspect='auto', cmap='YlOrRd',
                       vmin=0, vmax=clipped.max() or 1e-9)
        ax.set_xticks(range(RANK_CLIP))
        ax.set_xticklabels(range(1, RANK_CLIP + 1), fontsize=7)
        ax.set_yticks([0, 1, 2])
        ax.set_yticklabels(['Avg #1', 'Avg #2', 'Avg #3'], fontsize=9)
        ax.set_xlabel('Rank in individual seed', fontsize=9)
        ax.set_title(f'{MODEL_LABELS[model]}  ·  {METHOD_LABELS[method]}', fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
fname = os.path.join(FIG_DIR, 'arr_rankpos_heatmap.pdf')
fig.savefig(fname, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved: {fname}')


## 5. Selection churn vs rank churn

In [ ]:
# ── 6. Selection churn vs rank churn decomposition ────────────────────────────
# For each disagreeing (pair, instance):
#   (a) selection churn: differing feature NOT in the other seed's support set
#   (b) rank churn:      differing feature in BOTH support sets, only rank differs
# A disagreeing (pair, instance) event can belong to both categories.

EPS = 1e-10

sel_churn  = {mo: {} for mo in MODELS}
rank_churn = {mo: {} for mo in MODELS}

for model in MODELS:
    for method in METHODS:
        svs   = sv_data[model][method]
        min_n = min(len(v) for v in svs)
        n_sel = n_rank = n_disagree = 0

        for (si, sj) in PAIRS_10:
            vi = svs[si][:min_n]
            vj = svs[sj][:min_n]
            sup_i = np.abs(vi) > EPS   # (N, D) bool
            sup_j = np.abs(vj) > EPS
            top3_i = np.argsort(-np.abs(vi), axis=1)[:, :3]  # (N, 3)
            top3_j = np.argsort(-np.abs(vj), axis=1)[:, :3]

            for inst in range(min_n):
                set_i = set(top3_i[inst])
                set_j = set(top3_j[inst])
                if set_i == set_j:
                    continue
                n_disagree += 1
                differing = (set_i - set_j) | (set_j - set_i)
                has_sel = has_rank = False
                for feat in differing:
                    if sup_i[inst, feat] and sup_j[inst, feat]:
                        has_rank = True
                    else:
                        has_sel  = True
                if has_sel:  n_sel  += 1
                if has_rank: n_rank += 1

        denom = max(1, n_disagree)
        sel_churn[model][method]  = n_sel  / denom
        rank_churn[model][method] = n_rank / denom
        print(f'{model}/{method:<8}: n_disagree={n_disagree:4d}  '
              f'selection_churn={n_sel/denom:.1%}  rank_churn={n_rank/denom:.1%}')

# ── Plot grouped bars (overlapping events, not additive parts) ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
fig.suptitle(
    'Arrhythmia — Top-3 Churn Decomposition\n'
    '(conditional event frequencies; categories can overlap)',
    fontsize=12, fontweight='bold'
)

for ax, model in zip(axes, MODELS):
    x      = np.arange(len(METHODS))
    sel_v  = np.array([sel_churn[model][m]  for m in METHODS])
    rank_v = np.array([rank_churn[model][m] for m in METHODS])
    width = 0.36
    ax.bar(x - width/2, sel_v, width=width,  color='#E03131', edgecolor='white', linewidth=0.5, label='Selection churn')
    ax.bar(x + width/2, rank_v, width=width, color='#4393C3', edgecolor='white', linewidth=0.5, label='Rank churn')
    for xi, (sv, rv) in enumerate(zip(sel_v, rank_v)):
        if sv > 0.04:
            ax.text(xi - width/2, sv/2,      f'{sv:.1%}', ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
        if rv > 0.04:
            ax.text(xi + width/2, rv/2, f'{rv:.1%}', ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[m] for m in METHODS])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Fraction of Top-3 disagreements')
    ax.set_title(MODEL_LABELS[model])
    ax.legend(fontsize=9, loc='upper right')

fig.tight_layout()
fname = os.path.join(FIG_DIR, 'arr_churn_decomp.pdf')
fig.savefig(fname, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved: {fname}')


## 6. Summary paragraph

In [ ]:
# ── 7. Numeric summary paragraph ──────────────────────────────────────────────
N_TEST = sv_data['mlp']['random'][0].shape[0]

nonzero_avg = float(np.mean([
    (np.abs(sv_data[mo][mt][s]) > 1e-10).sum(axis=1).mean()
    for mo in MODELS for mt in METHODS for s in range(N_BG_SEEDS)
]))

# Model-averaged values for each metric
def mavg(d):
    return {mt: np.mean([d[mo][mt] for mo in MODELS]) for mt in METHODS}

avg_any  = mavg(top1_any)
avg_pair = mavg(top1_pair)
avg_sel  = np.mean([sel_churn[mo][mt]  for mo in MODELS for mt in METHODS])
avg_rank = np.mean([rank_churn[mo][mt] for mo in MODELS for mt in METHODS])

print('=== FULL TABLE ===')
print(f'{"":12}  {"any-seed":>10}  {"pairwise":>10}  {"sel-churn":>10}  {"rnk-churn":>10}')
for mo in MODELS:
    for mt in METHODS:
        print(f'{mo}/{mt:<8}:  {top1_any[mo][mt]:>9.1%}  {top1_pair[mo][mt]:>9.1%}  '
              f'{sel_churn[mo][mt]:>9.1%}  {rank_churn[mo][mt]:>9.1%}')

print()
print('=== PARAGRAPH DRAFT (fill in brackets) ===')
print(
    f'd={D_NOMINAL}, N={N_TEST} pilot: '
    f'top-1 any-seed churn {avg_any["random"]:.0%} (Random) / {avg_any["cte"]:.0%} (CTE); '
    f'pairwise {avg_pair["random"]:.0%} / {avg_pair["cte"]:.0%} (model-averaged). '
    f'l1_reg=\"num_features(10)\" concentrates mass on ~{nonzero_avg:.0f} effective features; '
    f'among top-3 disagreements, {avg_sel:.0%} are selection churn '
    f'(feature absent from the other seed\'s support) '
    f'and {avg_rank:.0%} are rank churn '
    f'(feature selected by both seeds but ranked differently).'
)
